# Comtrade Extraction — extending to 2024

Same pattern as `feature_engineering.ipynb` (`extract_products()` / `extract_all_streaming()` /
the `isAggregate`+bloc+zero-value cleanup from `GDELT_Extraction_full.ipynb`'s
`filter_comtrade_stream()`), pointed at a **new raw export file that includes 2024** and merged
into the existing 2017-2023 outputs rather than replacing them.

**Before running this:** `SRC_2024` below must point at a Comtrade bulk export that actually
contains `refYear == 2024` rows. The original `comtradeExports_updatedH5[240924]-wb.csv.gz`
(dated Sept 2024) predates full-year 2024 annual data being finalized, so it will NOT have
2024 in it — you need a fresher pull from UN Comtrade covering through 2024. If your export
already spans 2017-2024 in one file, set `SRC_2024` to that and skip the merge step at the end
(just run everything through `filter_comtrade_stream` once on the whole thing).

In [1]:
import pandas as pd
import numpy as np

# ---- config -- paths match tree_repo.txt's actual layout -----------------
import os
RAW_DIR = os.path.join("..", "..", "data", "raw")
PROCESSED_DIR = os.path.join("..", "..", "data", "processed")
# NOTE: data/raw/ currently only has comtradeExports_updatedH5[240924]-wb.csv.gz, which
# predates full-year 2024 data. You need to download a fresh export containing 2024 rows
# and place it in data/raw/ under whatever name UN Comtrade gives it -- update SRC_2024 below
# to match that actual filename before running this notebook.
SRC_2024 = os.path.join(RAW_DIR, "comtradeExports_2024_update.csv.gz")   # <-- rename to your actual downloaded file
OUT_ALL_2024 = os.path.join(PROCESSED_DIR, "all_products_2024.parquet")
CHUNK = 200_000

_KEEP = [
    "refYear", "reporterCode", "partnerCode", "cmdCode", "flowCode",
    "primaryValue", "FOBValue", "netWgt", "isAggregate",
    "pop_o", "gdp_o", "gdpcap_o", "pop_d", "gdp_d", "gdpcap_d", "dist",
]
_NUMERIC = ["primaryValue", "FOBValue", "netWgt",
            "pop_o", "gdp_o", "gdpcap_o", "pop_d", "gdp_d", "gdpcap_d", "dist"]
_GRAVITY = ["pop_o", "gdp_o", "gdpcap_o", "pop_d", "gdp_d", "gdpcap_d", "dist"]

## Generic extractor -- identical to `feature_engineering.ipynb`'s `extract_products()`

In [2]:
def extract_products(products, src=SRC_2024, out=None,
                     chunk=CHUNK, extra_cols=None, verbose=True):
    '''
    Extract Comtrade rows whose HS code starts with any of `products`.
    Same signature/behavior as the original in feature_engineering.ipynb.
    '''
    if products is None:
        products = [""]
    if isinstance(products, (str, int)):
        products = [products]
    prefixes = tuple(str(p) for p in products)

    available = pd.read_csv(src, nrows=0).columns.tolist()
    keep = [c for c in _KEEP if c in available]
    if extra_cols:
        keep += [c for c in extra_cols if c in available and c not in keep]
    numeric = [c for c in _NUMERIC if c in keep]
    gravity = [c for c in _GRAVITY if c in keep]

    if verbose:
        print(f"Extracting products starting with {list(prefixes)} from {src}")

    parts, total = [], 0
    reader = pd.read_csv(src, usecols=keep, chunksize=chunk, dtype=str, low_memory=False)
    for i, ch in enumerate(reader, 1):
        total += len(ch)
        code_col = ch["cmdCode"].astype(str).str.zfill(6)
        mask = code_col.str.startswith(prefixes[0])
        for p in prefixes[1:]:
            mask = mask | code_col.str.startswith(p)
        hit = ch[mask]
        if len(hit):
            parts.append(hit)
        if verbose and i % 10 == 0:
            print(f"  scanned {total:,}  ->  kept {sum(len(p) for p in parts):,}")

    if not parts:
        print("No rows matched those products.")
        return pd.DataFrame(columns=keep)

    df = pd.concat(parts, ignore_index=True)
    for c in numeric:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    z = df["cmdCode"].astype(str).str.zfill(6)
    df["chapter"], df["heading"] = z.str[:2], z.str[:4]

    if out:
        try:
            if out.endswith(".parquet"):
                df.to_parquet(out, index=False)
            else:
                df.to_csv(out, index=False, compression="gzip" if out.endswith(".gz") else None)
            if verbose:
                print(f"Saved -> {out}")
        except Exception as e:
            alt = out.rsplit(".", 1)[0] + ".csv.gz"
            df.to_csv(alt, index=False, compression="gzip")
            if verbose:
                print(f"(parquet failed: {e}) Saved -> {alt}")

    if verbose:
        print("\n" + "=" * 52)
        print(f"MATCHED {len(df):,} rows  (scanned {total:,})")
        print("=" * 52)
        print(f"Years: {sorted(df['refYear'].dropna().unique().tolist())}")
        print(f"Exporters: {df['reporterCode'].nunique()}   Importers: {df['partnerCode'].nunique()}")
        if "primaryValue" in df:
            print("\nprimaryValue (USD):")
            print(df['primaryValue'].describe().to_string())

    return df

## Streaming full-products extractor -- identical to the original's `extract_all_streaming()`

In [3]:
import pyarrow as pa, pyarrow.parquet as pq

def extract_all_streaming(src=SRC_2024, out=OUT_ALL_2024, chunk=CHUNK):
    keep = [c for c in _KEEP if c in pd.read_csv(src, nrows=0).columns]
    writer, total = None, 0
    for i, ch in enumerate(pd.read_csv(src, usecols=keep, chunksize=chunk,
                                       dtype=str, low_memory=False), 1):
        total += len(ch)
        for c in _NUMERIC:
            if c in ch: ch[c] = pd.to_numeric(ch[c], errors="coerce")
        z = ch["cmdCode"].astype(str).str.zfill(6)
        ch["chapter"], ch["heading"] = z.str[:2], z.str[:4]
        table = pa.Table.from_pandas(ch, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(out, table.schema)
        writer.write_table(table)
        if i % 20 == 0:
            print(f"  written {total:,} rows")
    if writer:
        writer.close()
    print(f"DONE -> {out}  ({total:,} rows)")

extract_all_streaming()

FileNotFoundError: [Errno 2] No such file or directory: '../../data/raw/comtradeExports_2024_update.csv.gz'

In [4]:
import pandas as pd

SRC = "../../data/raw/comtradeExports_updatedH5[240924]-wb.csv.gz"

years_seen = set()
for i, chunk in enumerate(pd.read_csv(SRC, usecols=["refYear"], chunksize=500_000, dtype=str), 1):
    years_seen.update(chunk["refYear"].dropna().unique().tolist())
    if i % 10 == 0:
        print(f"  scanned chunk {i}, years so far: {sorted(years_seen)}")

print("\nFINAL year range in this file:", sorted(years_seen))

  scanned chunk 10, years so far: ['2022']
  scanned chunk 20, years so far: ['2022', '2023']
  scanned chunk 30, years so far: ['2022', '2023']
  scanned chunk 40, years so far: ['2017', '2022', '2023']
  scanned chunk 50, years so far: ['2017', '2019', '2022', '2023']
  scanned chunk 60, years so far: ['2017', '2019', '2022', '2023']
  scanned chunk 70, years so far: ['2017', '2019', '2021', '2022', '2023']
  scanned chunk 80, years so far: ['2017', '2019', '2020', '2021', '2022', '2023']
  scanned chunk 90, years so far: ['2017', '2019', '2020', '2021', '2022', '2023']
  scanned chunk 100, years so far: ['2017', '2018', '2019', '2020', '2021', '2022', '2023']
  scanned chunk 110, years so far: ['2017', '2018', '2019', '2020', '2021', '2022', '2023']

FINAL year range in this file: ['2017', '2018', '2019', '2020', '2021', '2022', '2023']


## Sanity check before cleaning/merging

Confirms 2024 actually landed, and only 2024 (or whatever your `SRC_2024` covers) --
same style of check as the original notebook's post-extraction summaries.

In [ ]:
import pyarrow.parquet as pq
pf = pq.ParquetFile(OUT_ALL_2024)
print("rows:", pf.metadata.num_rows, "| row groups:", pf.num_row_groups)
s = pf.read_row_group(0).to_pandas()
print("years in first row group:", sorted(pd.to_numeric(s['refYear'], errors='coerce').dropna().unique().tolist()))

## Clean this year's slice the same way `filter_comtrade_stream()` did for 2017-2023

Same three rules, same order, same Taiwan carve-out (490 kept, other bloc codes dropped) --
`country_codes.DROP_CODES` already encodes that.

In [ ]:
from country_codes import DROP_CODES

def filter_comtrade_stream(src, out):
    pf = pq.ParquetFile(src)
    writer, total, kept = None, 0, 0
    for b in range(pf.num_row_groups):
        ch = pf.read_row_group(b).to_pandas()
        total += len(ch)
        ch = ch[ch["isAggregate"].astype(str) == "0"]
        ch = ch[~ch["reporterCode"].astype(str).isin(DROP_CODES)]
        ch = ch[~ch["partnerCode"].astype(str).isin(DROP_CODES)]
        ch["primaryValue"] = pd.to_numeric(ch["primaryValue"], errors="coerce")
        ch = ch[ch["primaryValue"] > 0]
        if ch.empty:
            continue
        kept += len(ch)
        table = pa.Table.from_pandas(ch, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(out, table.schema)
        writer.write_table(table)
        if b % 25 == 0:
            print(f"  row-group {b}/{pf.num_row_groups} | kept {kept:,}/{total:,}")
    if writer:
        writer.close()
    print(f"DONE -> {out}  (kept {kept:,} of {total:,} = {100*kept/total:.1f}%)")

filter_comtrade_stream(OUT_ALL_2024, os.path.join(PROCESSED_DIR, "all_products_2024_ready.parquet"))

## Merge with the existing 2017-2023 file

Produces the combined `all_products_ready_2017_2024.parquet` that `train_benchmark.py`'s
`load_or_split()` should point at for the extended train-through-2023/test-2024 split.
Dedupes defensively on the natural key in case of any overlap.

In [ ]:
existing = pd.read_parquet(os.path.join(PROCESSED_DIR, "all_products_ready.parquet"))   # your current 2017-2023 file
new_2024 = pd.read_parquet(os.path.join(PROCESSED_DIR, "all_products_2024_ready.parquet"))

print("existing years:", sorted(existing['refYear'].unique()))
print("new file years:", sorted(new_2024['refYear'].unique()))

combined = pd.concat([existing, new_2024], ignore_index=True)
combined = combined.drop_duplicates(
    subset=["refYear", "reporterCode", "partnerCode", "cmdCode"], keep="last")

MERGED_OUT_2024 = os.path.join(PROCESSED_DIR, "all_products_ready_2017_2024.parquet")
combined.to_parquet(MERGED_OUT_2024, index=False)
print(f"\nCombined: {len(combined):,} rows, years {sorted(combined['refYear'].unique())} -> {MERGED_OUT_2024}")